# DSCI 100 Final Project Report

## (1) Introduction:

A research group in Computer Science at UBC, led by Dr. Frank Wood, is gathering data on video game behavior through a Minecraft server. Their players.csv dataset contains 196 unique observations with 7 variables that capture demographic information and player skill levels as follows:

- **experience**: Describes the player's level of gaming experience. (*fct, loaded as chr*)
- **subscribe**: Indicates whether the player is subscribed to the game's associated newsletter. (*lgl*)
- **hashedEmail**: The player's email encrypted as a code. (*chr*)
- **played_hours**: The number of hours the player has played on the Minecraft server. (*dbl*)
- **name**: The player's first name. (*chr*)
- **gender**: The player's gender. (*fct, loaded as chr*) 
- **Age**: The player's age in years. (*int, loaded as double*)

Our project investigates which player characteristics and behaviors predict newsletter subscription and how these predictors differ among various player types. Specifically, we ask: can a player's age (`Age`) and number of hours player on the server (`played_hours`) predict whether they are subscribed to the newsletter, according to the `players` dataset? This question supports Dr. Frank Wood's research by helping to target recruitment efforts toward individuals likely to subscribe. Their willingness to subscribe suggests they may be more engaged and responsive, making them better subjects for ongoing studies. Thus, identifying the predictors that reveal which types of players have an increased tendency to subscribe will enable better recruitment. Unfortunately, it is unclear if data submission was mandatory for all players; if only a subset contributed, the dataset may not represent the entire playerbase, limiting its usefulness for building an effective classifier. We also assume that the demographic (especially age) players provided are truthful.

## (2) Methods:

Predictions will be built off of the `players` data from the players.csv file. 

Since the response variable (`subscribe`) is categorical, we will use K-nearest neighbors (KNN) classification to predict whether a player is subscribed based on `Age` and `played_hours`. While the model only relies on two predictors, this may not be a limitation, as adding more variables does not necessarily improve classifier performance. 

To prepare the data, we will convert `subscribe` to a factor and drop any observations with missing values for ease of handling.

We will split 75% of the data into a training set and reserve the remaining 25% as a test set, giving the model sufficient data to learn on while still leaving enough unseen data for a reliable performance evaluation. Both `Age` and `played_hours` will be centered and scaled so that they contribute equally to the Euclidean distance calculations used by KNN. We will use 5-fold cross-validation on the training set to determine the optimal K value based on accuracy, as relying on a single split can produce misleading results if it happens to be unrepresentative of the overall data. 5-fold is fairly computationally efficient and does not produce very small training subsets that can result in unstable performance estimates. We will test K values from 1 to 10 because the dataset is relatively small, meaning larger K values are likely to underfit it. After identifying the best K, we will finalize the model using that value and evaluate its prediction performance on the test set via accuracy, precision, and recall.

## (3) Code and Results:

In [ ]:
library(tidyverse)
library(tidymodels)
library(cowplot)
library(repr)

set.seed(4)
options(repr.plot.width = 14)

players_url <- "https://raw.githubusercontent.com/oo74/DSCI-100-Project/d932a95bab3bbe9a443dcba02939882b0735483f/data/players.csv"
players <- read_csv(players_url) |>
    mutate(subscribe = fct_recode(as_factor(subscribe), Yes = "TRUE", No = "FALSE"))


nrow(players)

players |>
    map_df(n_distinct)

players |>
    group_by(subscribe) |>
    summarize(count = n())


players |>
    summarize(mean = mean(played_hours, na.rm = TRUE),
              SD = sd(played_hours, na.rm = TRUE),
              min = min(played_hours, na.rm = TRUE),
              max = max(played_hours, na.rm = TRUE),
              median = median(played_hours, na.rm = TRUE))

players |>
    summarize(mean = mean(Age, na.rm = TRUE),
              SD = sd(Age, na.rm = TRUE),
              min = min(Age, na.rm = TRUE),
              max = max(Age, na.rm = TRUE),
              median = median(Age, na.rm = TRUE))

The `played_hours` variable spans a wide range—from 0 to 223.1 hours—with a standard deviation of 28.36 hours, indicating considerable variability; however, a mean of 5.85 hours and a median of 0.1 hours suggest that most players have very few recorded hours. It is unclear whether these low hours are a consequence of the players being new or a genuine lack of interest in gaming. This distinction is important for predictions based on `played_hours`, as this metric may not accurately capture a player's typical gaming behavior. Though `Age` ranges from 8 to 50 years, the mean of 20.5 years, the median of 19 years, and the modest standard deviation of 6.17 indicates that most players are relatively young.

| Variable Name | No. of Unique Values | Mean | Standard Deviation | Min | Max | Median |
| -------- | ------- | ------- | ------- | ------- | ------- | ------- |
| subscribe | 2 |
| played_hours | 43 | 5.845918 |28.35734 | 0 | 223.1 | 0.1 |
| Age | 31 | 20.52062 | 6.174667 | 8 | 50 | 19 |

The `subscribe` variable was converted from a logical to a factor type using `fct_recode()` for proper KNN classification labeling. The data includes 142 newsletter subscribers and 52 non-subscribers, indicating imbalanced classes for `subscribe`. 

In [ ]:
age_hours_plot <- players |>
    ggplot(aes(x = Age, y = played_hours, color = subscribe)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 1: Age, played hours, and actual subscription status of all players.", color = "Subscribed?") +
    theme(text = element_text(size = 16), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

age_plot <- players |>
    ggplot(aes(x = Age, fill = subscribe)) +
    geom_histogram(binwidth = 1) +
    labs(x = "Age (years)", y = "Count", title = "Fig. 2: Distribution of age of players and their corresponding subscription status.") +
    theme(text = element_text(size = 10), legend.position = "bottom", legend.direction = "horizontal") +
    scale_fill_manual(values = c("red", "chartreuse3"))

hours_plot <- players |>
    ggplot(aes(x = played_hours, fill = subscribe)) +
    geom_histogram(binwidth = 2) +
    labs(x = "Played Hours", y = "Count", title = "Fig. 3: Distribution of players' played hours and their corresponding subscription status.") +
    theme(text = element_text(size = 10), legend.position = "bottom", legend.direction = "horizontal") +
    scale_fill_manual(values = c("red", "chartreuse3"))


age_hours_plot
plot_grid(age_plot, hours_plot, ncol = 2)

To reduce overplotting and enhance density visualization, datapoint opacity was set to 0.6, where higher opacity indicates more overlap. 

Fig. 1 shows no clear linear relationship between hours played and age. Most players are clustered around zero hours, with a few outliers exceeding 150 hours. As seen in Fig. 2, the majority of players are between 15 and 28 years old, with a significant concentration at age 17. Nearly all age groups 17 and older contain non-subscribed players, while those younger are all subscribed. Fig. 3 indicates that most non-subscribed players have low played hours. However, since the majority of all players report low hours and there is limited data for high-hour players, this observation may not imply a direct association between low played hours and the tendency to subscribe.

In [ ]:
players <- drop_na(players)
nrow(players)

k_vals <- tibble(neighbors = 1:10)

players_split <- initial_split(players, prop = 0.75, strata = subscribe)
players_train <- training(players_split)
players_test <- testing(players_split)

players_recipe <- recipe(subscribe ~ played_hours + Age, data = players_train) |>
    step_center(all_predictors()) |>
    step_scale(all_predictors()) 

players_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
    set_engine("kknn") |>
    set_mode("classification")

vfold_sets <- players_train |>
    vfold_cv(v = 5, strata = subscribe)

k_accuracies <- workflow() |>
    add_recipe(players_recipe) |>
    add_model(players_spec) |>
    tune_grid(resamples = vfold_sets, grid = k_vals) |>
    collect_metrics() |>
    filter(.metric == "accuracy") |>
    mutate(accuracy = mean) |>
    select(neighbors, accuracy)
    
k_accuracies_plot <- k_accuracies |>
    ggplot(aes(x = neighbors, y = accuracy)) +
    geom_point() +
    geom_line() +
    labs(x = "Neighbors (K)", y = "Accuracy", title = "Fig. 4: Accuracies associated with various K values.") +
    theme(text = element_text(size = 12))

best_k <- k_accuracies |>
    slice_max(accuracy) |>
    slice_min(neighbors) |>
    pull(neighbors)

best_k
k_accuracies_plot

The 2 observations with `NA` were removed to avoid interference with classification. This small loss of data is unlikely to have a significant impact on the predictions.

Before modeling, 75% of the data was split into a training set and 25% was split into a testing set. As mentioned in the methods section, this ensures that we evaluate model performance on unseen data rather than on the training set, thereby avoiding inflated metrics and poor generalization for new observations. The recipe's formula defined the label and predictors while standardizing the predictors. We then built the model as a KNN classifier.

We determined the best K using 5-fold cross-validation by selecting the K with the highest accuracy. In cases of ties for highest accuracy, we chose the smallest K for computational efficiency. As shown in Fig. 4, mean accuracy increased until K reached 7 and then decreased, suggesting that 7 is the best K within the tested range.

In [ ]:
players_best_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = best_k) |>
    set_engine("kknn") |>
    set_mode("classification")

players_fit <- workflow() |>
    add_recipe(players_recipe) |>
    add_model(players_best_spec) |>
    fit(data = players_train)

players_predicted <- players_fit |>
    predict(players_test) |>
    bind_cols(players_test) 

players_plot <- players_predicted |>
    ggplot(aes(x = Age, y = played_hours, color = subscribe)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 5: Age, played hours, and actual subscription status of players in test set.", color = "Subscribed?") +
    ylim(0, 2) +
    theme(text = element_text(size = 11), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

players_predicted_plot <- players_predicted |>
    ggplot(aes(x = Age, y = played_hours, color = .pred_class)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 6: Age, played hours, and predicted subscription status of players in test set", color = "Predicted to subscribe?") +
    ylim(0, 2) +
    theme(text = element_text(size = 11), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

plot_grid(players_plot, players_predicted_plot, ncol = 2)

With the optimal K determined, a new model was built using that K to predict outcomes on the test set. Since initial visualizations revealed that most observations reported under 2 player hours, the y-axis was capped at 2 to spread out the majority of the datapoints and enhance clarity.

In [ ]:
players_accuracy <- players_predicted |>
    metrics(truth = subscribe, estimate = .pred_class) |>
    filter(.metric == "accuracy")

players_precision <- players_predicted |>
    precision(truth = subscribe, estimate = .pred_class, event_level = "second")

players_recall <- players_predicted |>
    recall(truth = subscribe, estimate = .pred_class, event_level = "second")

players_conf_mat <- players_predicted |>
    conf_mat(truth = subscribe, estimate = .pred_class)

players_conf_mat
bind_rows(players_accuracy, players_precision, players_recall) |>
    select(-.estimator)

Model performance metrics were calculated with "Yes" as the positive class for the variable `subscribe`. The model achieved an accuracy of 0.7142857, a recall of 0.7777778, and a precision of 0.8235294. The confusion matrix and Fig. 6 suggest that the model was more likely to predict "Yes" than "No".

## (4) Discussion:
- summarize what you found
- discuss whether this is what you expected to find?
- discuss what impact could such findings have?
- discuss what future questions could this lead to?

Our KNN classifier, which used `Age` and `played_hours` as predictors, achieved an accuracy of 0.71, a precision of 0.82, and a recall of 0.78. While these numbers might seem moderately high, an accuracy of 0.71 means nearly 3 in every 10 individuals are misclassified, indicating that the classifier is not very good. 

The high recall indicates that any positive data in the test data will likely be found by classifier, and the high precision indicates that a prediction of positive by classifier is very likely positive. However, just because they are high doesn't necessarily mean that the classifier is good; it could be a result of the class imbalance, where most of the data is "Yes" for `subscribe`. 




The precision of 0.82 implies that when the model predicts a subscription, it is correct 82% of the time; however, this metric may be inflated due to class imbalance, where the model's tendency to predict "Yes" (reflected in a recall of 0.78) arises from having few "No" cases in the dataset.

To address this imbalance, upsampling the "No" class during pre-processing could help balance the data and yield a clearer assessment of the predictors' effectiveness. If upsampling shows that age and played_hours are good indicators of subscription, the model could prove useful for guiding marketing efforts. Conversely, if these predictors remain weak, it may suggest the need to explore additional variables.

Furthermore, there is simply not enough data—only 194 observations remain after dropping missing values, with even fewer available for training (75% of the data), particularly among older demographics and high playtime users. While adding more predictors might improve performance, the dataset offers few quantitative variables. For example, the categorical variable experience (amateur, beginner, regular, pro, veteran) could be recoded numerically, but the resulting numbers may not accurately reflect the true "distance" between experience levels (e.g., the gap between "amateur" and "regular" may not be equivalent to that between other levels).

Initially, we expected that a higher number of played hours and a lower age would indicate a higher likelihood of subscribing, as more engaged, younger players might be more interested in gaming newsletters. However, our findings suggest that these predictors alone may not be sufficient to accurately determine subscription status, prompting further investigation into additional factors or data balancing techniques.






The model's moderately high recall could be a result of its increased tendency to predict "Yes" over "No" for `subscribe`. This could result from class imbalance; with few "No" in the data, even if the model classifies non-subscribed players as subscribed, accuracy and precision remain inflated. To address this, upsampling the "No" class during pre-processing could balance the dataset to help this. If upsampling shows that age and number of hours played are good indicators of a players subscription status, then this could help the developers of the game with their marketing efforts. However, if analysis of data with upsampling still indicates age and number of hours played as poor indicators of subscription status, then this could raise further questions about what other variables may be indicative of subscription status then. (briefly interpret the precision and what it says about the model)

Initially, we expected that a higher number of played hours and a lower age would indicate subscription, as more engaged, younger players might show greater interest in gaming newsletters. However, the inadequate metrics obtained from the model suggest these predictors may not be sufficient to accurately predict subscription status.

Furthermore, there is simply not enough data (only 194 observations after NAs dropped), especially for a lot of the older demographics and those with high playtime, so the poor prediction could also be impacted by the lacking data. Perhaps, as an improvement, other predictors could be added, but this can be difficult to do given that addition of predictors does not necessraily mean more accurate, and the players dataset does not provide many quantitative variables to use as predictors. THe 'experience' categorical variable (amateur, beginner, regular, pro, veteran) could be recoded to numerical values, but the assigned numbers may not accurately reflect the true "distance" between levels (e.g., the gap between "amateur" and "regular" may not be proportional to that between other levels).
